In [0]:
import pyspark.sql.functions as F

df = spark.table("workspace.erp_lakehouse.bronze_b_items")
df.display()

# change all the uom value to a uppoer case. 
df = df.withColumn("uom", F.upper(F.col("uom")))
df.display()


In [0]:
# Data cleaning
# Update EACH to EA since they mean the same thing. This will prevent issues when grouping data.
#
df = df.withColumn("uom", F.when(df.uom == 'EACH', 'EA').otherwise(df.uom))
df.display()


In [0]:
# casing the list_price the data need to be change to a desimal to do math
# Data cleaning
# Update the data to be in the correct format
print(df.schema['list_price'].dataType)
df = df.withColumn("list_price", F.expr("try_cast(list_price AS DECIMAL(10,2))"))
df.display()
print(f"after update {df.schema['list_price'].dataType}")

In [0]:
# Remove stray spaces so the same value isn't treated as two different values
df = (df
    .withColumn("part_no", F.trim(F.col("part_no")))
    .withColumn("uom", F.trim(F.col("uom"))))

In [0]:
from pyspark.sql.window import Window

# Same part_no twice: keep the row with the higher price
w = Window.partitionBy("part_no").orderBy(F.col("list_price").desc())
print(df.count())
items_silver = (df
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .drop("rn"))
items_silver.display()

print(items_silver.count(), items_silver.select("part_no").distinct().count())

df.createOrReplaceTempView("items_df")
items_silver.createOrReplaceTempView("items_silver")


In [0]:
%sql

SELECT j.part_no, j.list_price,i.part_no, i.list_price
FROM items_df j
LEFT JOIN items_silver i
  ON j.part_no = i.part_no
 AND j.list_price = i.list_price
WHERE i.part_no IS NULL;

In [0]:
items_silver.groupBy("uom").count().display()   # only EA should show
print(items_silver.count(), items_silver.select("part_no").distinct().count())   # both numbers must match

In [0]:
items_silver.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.erp_lakehouse.silver_b_items")

In [0]:
dfs = spark.table("workspace.erp_lakehouse.silver_b_items")
display(dfs)

Working on the orders

In [0]:
SCHEMA = "workspace.erp_lakehouse"
orders = spark.table(f"{SCHEMA}.bronze_b_orders")
orders.display()
#

In [0]:
%sql
SELECT * FROM  workspace.erp_lakehouse.bronze_b_orders bbo
WHERE bbo.cust_id IS NULL

There is 6 orders that dose have cust_id ![image_1794242061028.png](../docs/images/image_1794242061028.png)
In addition to that, the order_dt is in two different formats.

In [0]:
orders = orders.withColumn("order_date", F.coalesce(
    F.expr("try_to_date(order_dt, 'yyyy-MM-dd')"),
    F.expr("try_to_date(order_dt, 'MM/dd/yyyy')")
))

# check if the conversion was successful
orders.select("order_dt", "order_date").display()
orders.filter("order_date is null").count()

In [0]:
#  same procces for ship_dt
orders = orders.withColumn("ship_date", F.coalesce(
    F.expr("try_to_date(ship_dt, 'yyyy-MM-dd')"),
    F.expr("try_to_date(ship_dt, 'MM/dd/yyyy')")
))

# check if the conversion was successful
orders.select("ship_dt", "ship_date").display()
orders.filter("ship_date is null").count()

In [0]:
# Flag rows that fail validation so they can be quarantined later
orders = orders.withColumn("reject_reason",
    F.when(F.col("cust_id").isNull(), "missing cust_id")
     .when(F.col("order_date").isNull(), "bad order date"))

In [0]:
# Split orders: valid rows go to silver, rejected rows are quarantined
orders_silver = orders.filter("reject_reason IS NULL").drop("reject_reason")
orders_quarantine = orders.filter("reject_reason IS NOT NULL")
orders_quarantine.display()

![image_1794246232034.png](../docs/images/image_1794246232034.png)


In [0]:
orders_silver_count = orders_silver.count()
orders_quarantine_count = orders_quarantine.count()
bronze_orders_count = spark.table("workspace.erp_lakehouse.bronze_b_orders").count()

if orders_silver_count + orders_quarantine_count != bronze_orders_count:
  raise Exception("Data was not properly split between silver and quarantine")
else:
    print("Data was properly split between silver and quarantine")
    print(f"Valid orders: {orders_silver_count} | Rejected orders: {orders_quarantine_count} |  total orders: {bronze_orders_count}")



Orders are done next will be order line.